# IEX Data Cleaner

Notebook này sẽ xử lý file `iex-data.xlsx` để đưa về dạng chuẩn: Agent | Activity | Start | End.
- Xử lý phần header dư thừa
- Chuẩn hóa tên cột
- Chuyển Start/End về datetime
- Xuất ra file sạch `iex_clean.csv`


In [49]:
import pandas as pd

# Hiển thị đủ số dòng và số cột mong muốn
pd.set_option("display.max_rows", 100)   # tối đa số dòng hiển thị
pd.set_option("display.max_columns", None)  # hiện tất cả các cột
pd.set_option("display.width", None)   # không giới hạn độ rộng

# Đọc file, bỏ 5 dòng đầu tiên
iex_df = pd.read_excel("iex-data.xlsx", skiprows=5)

# Bỏ những dòng toàn NaN
iex_df = iex_df.dropna(how="all")

# Xóa các cột toàn NaN
iex_df = iex_df.dropna(axis=1, how="all")

# Reset lại index
iex_df = iex_df.reset_index(drop=True)

# Xem thử 50 dòng đầu tiên
iex_df.head(50)

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,MU: 1334 Expedia ENG Ho Chi Minh VNM,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 10
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
2,NaN,8/21/25,Off,NaN,NaN,NaN,NaN,NaN
3,"Agent: 3085733 BUI, MINHAN",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
5,NaN,8/21/25,6:00 AM,3:00 PM,NaN,Open Time,6:00 AM,8:30 AM
6,NaN,NaN,NaN,NaN,NaN,Break,8:30 AM,8:45 AM
7,NaN,NaN,NaN,NaN,NaN,Open Time,8:45 AM,9:30 AM
8,NaN,NaN,NaN,NaN,NaN,Break,9:30 AM,9:45 AM
9,NaN,NaN,NaN,NaN,NaN,Open Time,9:45 AM,1:00 PM


In [50]:
# Gán lại tên cột thành số (0, 1, 2, ...)
iex_df.columns = range(iex_df.shape[1])

# Xem thử 20 dòng đầu
iex_df.head(20)

,0,1,2,3,4,5,6,7
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
2,NaN,8/21/25,Off,NaN,NaN,NaN,NaN,NaN
3,"Agent: 3085733 BUI, MINHAN",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End
5,NaN,8/21/25,6:00 AM,3:00 PM,NaN,Open Time,6:00 AM,8:30 AM
6,NaN,NaN,NaN,NaN,NaN,Break,8:30 AM,8:45 AM
7,NaN,NaN,NaN,NaN,NaN,Open Time,8:45 AM,9:30 AM
8,NaN,NaN,NaN,NaN,NaN,Break,9:30 AM,9:45 AM
9,NaN,NaN,NaN,NaN,NaN,Open Time,9:45 AM,1:00 PM


In [51]:
import re

# Hàm tách IEX Id và Name từ chuỗi
def split_agent_info(value):
    if isinstance(value, str) and value.startswith("Agent:"):
        # Loại bỏ "Agent:" rồi strip
        cleaned = value.replace("Agent:", "").strip()
        # Dùng regex tách id (số đầu tiên) và phần còn lại (tên)
        match = re.match(r"(\d+)\s+(.+)", cleaned)
        if match:
            return match.group(1), match.group(2)
    return None, None

# Tách thông tin Agent
iex_df["IEX Id"], iex_df["Name"] = zip(*iex_df[0].map(split_agent_info))

# Kiểm tra kết quả
iex_df.head(20)

,0,1,2,3,4,5,6,7,IEX Id,Name
0,"Agent: 3052306 BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN,3052306,"BUI, BADUONG"
1,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End,None,None
2,NaN,8/21/25,Off,NaN,NaN,NaN,NaN,NaN,None,None
3,"Agent: 3085733 BUI, MINHAN",NaN,NaN,NaN,NaN,NaN,NaN,NaN,3085733,"BUI, MINHAN"
4,NaN,Date,Start,End,Scheduled Activity,NaN,Start,End,None,None
5,NaN,8/21/25,6:00 AM,3:00 PM,NaN,Open Time,6:00 AM,8:30 AM,None,None
6,NaN,NaN,NaN,NaN,NaN,Break,8:30 AM,8:45 AM,None,None
7,NaN,NaN,NaN,NaN,NaN,Open Time,8:45 AM,9:30 AM,None,None
8,NaN,NaN,NaN,NaN,NaN,Break,9:30 AM,9:45 AM,None,None
9,NaN,NaN,NaN,NaN,NaN,Open Time,9:45 AM,1:00 PM,None,None


In [52]:
# 1. Xóa cột 0
iex_df = iex_df.drop(columns=[0])

# 2. Đưa IEX Id và Name ra ngoài cùng bên trái
cols = ["IEX Id", "Name"] + [col for col in iex_df.columns if col not in ["IEX Id", "Name"]]
iex_df = iex_df[cols]

# 3. Nếu cột 2 = "Off" thì cột 3 = "Off"
iex_df.loc[iex_df[2] == "Off", 3] = "Off"

# Xem kết quả
iex_df.head(20)

,IEX Id,Name,1,2,3,4,5,6,7
0,3052306,"BUI, BADUONG",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
2,None,None,8/21/25,Off,Off,NaN,NaN,NaN,NaN
3,3085733,"BUI, MINHAN",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
5,None,None,8/21/25,6:00 AM,3:00 PM,NaN,Open Time,6:00 AM,8:30 AM
6,None,None,NaN,NaN,NaN,NaN,Break,8:30 AM,8:45 AM
7,None,None,NaN,NaN,NaN,NaN,Open Time,8:45 AM,9:30 AM
8,None,None,NaN,NaN,NaN,NaN,Break,9:30 AM,9:45 AM
9,None,None,NaN,NaN,NaN,NaN,Open Time,9:45 AM,1:00 PM


In [53]:
# Tạo cột Shift
def get_shift(row):
    # Nếu cột 1 (Name) bị NaN hoặc bằng "Date" thì bỏ qua
    if pd.isna(row[1]) or str(row[1]).strip().lower() == "date":
        return None
    # Nếu cả Start và End đều là Off
    elif str(row[2]).strip().lower() == "off" and str(row[3]).strip().lower() == "off":
        return "Off"
    # Nếu có giờ Start và End
    elif pd.notna(row[2]) and pd.notna(row[3]):
        return f"{row[2]} - {row[3]}"
    else:
        return None

iex_df["Shift"] = iex_df.apply(get_shift, axis=1)

# Đưa cột Shift ngay sau cột Name
cols = list(iex_df.columns)
name_index = cols.index("Name")
cols.insert(name_index + 1, cols.pop(cols.index("Shift")))
iex_df = iex_df[cols]

iex_df.head(50)


,IEX Id,Name,Shift,1,2,3,4,5,6,7
0,3052306,"BUI, BADUONG",None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
2,None,None,Off,8/21/25,Off,Off,NaN,NaN,NaN,NaN
3,3085733,"BUI, MINHAN",None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,None,Date,Start,End,Scheduled Activity,NaN,Start,End
5,None,None,6:00 AM - 3:00 PM,8/21/25,6:00 AM,3:00 PM,NaN,Open Time,6:00 AM,8:30 AM
6,None,None,None,NaN,NaN,NaN,NaN,Break,8:30 AM,8:45 AM
7,None,None,None,NaN,NaN,NaN,NaN,Open Time,8:45 AM,9:30 AM
8,None,None,None,NaN,NaN,NaN,NaN,Break,9:30 AM,9:45 AM
9,None,None,None,NaN,NaN,NaN,NaN,Open Time,9:45 AM,1:00 PM


In [54]:
# Downfill IEX Id và Name trước
iex_df["IEX Id"] = iex_df["IEX Id"].ffill()
iex_df["Name"] = iex_df["Name"].ffill()

# Shift: vừa upfill vừa downfill trong phạm vi từng IEX Id
iex_df["Shift"] = (
    iex_df.groupby("IEX Id")["Shift"]
    .transform(lambda x: x.ffill().bfill())
)


# Xử lý cột 1 (ngày): thay "Date" bằng NaN, rồi upfill + downfill trong phạm vi IEX Id
iex_df[1] = iex_df[1].replace("Date", pd.NA)
iex_df[1] = (
    iex_df.groupby("IEX Id")[1]
    .transform(lambda x: x.ffill().bfill())
)

# Đổi tên cột 1 thành "Date"
iex_df = iex_df.rename(columns={1: "Date"})

# Chuẩn hoá format ngày (YYYY-MM-DD)
iex_df["Date"] = pd.to_datetime(iex_df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Remove cột 2,3 (giữ lại cột "Date")
iex_df = iex_df.drop(columns=[2, 3])

# Remove dòng Scheduled Activity
iex_df = iex_df[iex_df[4] != "Scheduled Activity"]

# Remove dòng mà 5,6,7 đều NaN
iex_df = iex_df.dropna(subset=[5, 6, 7], how="all")

iex_df


,IEX Id,Name,Shift,Date,4,5,6,7
5,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,NaN,Open Time,6:00 AM,8:30 AM
6,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,NaN,Break,8:30 AM,8:45 AM
7,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,NaN,Open Time,8:45 AM,9:30 AM
8,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,NaN,Break,9:30 AM,9:45 AM
9,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,NaN,Open Time,9:45 AM,1:00 PM
...,...,...,...,...,...,...,...,...
1318,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-21,NaN,Open Time,6:15 AM,8:40 AM
1319,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-21,NaN,Lunch,8:40 AM,9:40 AM
1320,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-21,NaN,Open Time,9:40 AM,12:25 PM
1321,3089155,"Vuu, TienNhan",5:00 AM - 2:00 PM,2025-08-21,NaN,Break,12:25 PM,12:40 PM


In [55]:
# Xóa cột 4
iex_df = iex_df.drop(columns=[4])

# Đổi tên cột 5,6,7
iex_df = iex_df.rename(columns={
    5: "Activity",
    6: "Start time",
    7: "End time"
})

iex_df.head(20)

,IEX Id,Name,Shift,Date,Activity,Start time,End time
5,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Open Time,6:00 AM,8:30 AM
6,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Break,8:30 AM,8:45 AM
7,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Open Time,8:45 AM,9:30 AM
8,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Break,9:30 AM,9:45 AM
9,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Open Time,9:45 AM,1:00 PM
10,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Lunch,1:00 PM,2:00 PM
11,3085733,"BUI, MINHAN",6:00 AM - 3:00 PM,2025-08-21,Open Time,2:00 PM,3:00 PM
17,3097227,"Bui, HoangCamNhung",5:00 AM - 2:00 PM,2025-08-21,Open Time,5:00 AM,6:15 AM
18,3097227,"Bui, HoangCamNhung",5:00 AM - 2:00 PM,2025-08-21,Break,6:15 AM,6:30 AM
19,3097227,"Bui, HoangCamNhung",5:00 AM - 2:00 PM,2025-08-21,Open Time,6:30 AM,10:40 AM


In [56]:
# Xuất ra file Excel, nếu đã có sẵn thì sẽ ghi đè
iex_df.to_excel("iex-data-extracted.xlsx", index=False)